### Exercise 1: The Self-Correction Loop

Topic: Structured Output & Pydantic Validation


Goal: Handle cases where the LLM provides the wrong data type for a schema field.

In [ ]:
from langchain.chat_models import init_chat_model
import os
from pydantic import BaseModel, Field
from langchain.messages import HumanMessage

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")

class UserExtractor(BaseModel):
    name: str = Field(description="The name of the user")
    age: int = Field(description="The age of the user")

struct_model = model.with_structured_output(UserExtractor)

def extract_user_info(text: str) -> UserExtractor:
    try:
        response = struct_model.invoke(text)
        return response
    except Exception as e:
        correction_prompt = HumanMessage(
            content=f"""
You provided an invalid format.
Error:
{e}
Please extract the information again and ensure the age is a numeric integer.
Text:
{text}
""")
        # Retry the extraction with the correction prompt
        corrected_response = struct_model.invoke(correction_prompt)
        return corrected_response

output = extract_user_info("My name is Alice and I am twenty-five years old.")
print(output)
    





name='Alice' age=25
